# Scopus — base de publicaciones por autor (libreta consolidada)

Extrae publicaciones (2019–2025) de un roster de profesores usando el **Scopus
Search API** y enriquece autores + afiliaciones con **OpenAlex** (gratis, por DOI).

Salida: `scopus_out/scopus_publications.csv` — **una fila por publicación**, limpia
(UTF-8, sin mojibake), con: profesores rastreados, ids, título, fecha, autores y afiliaciones.

> El token se carga desde `config.py` (ignorado por git). La primera vez:
> `cp config.example.py config.py` y coloca tu `ELSEVIER_API_KEY`.

In [ ]:
# Si hace falta, instala dependencias:
# !pip install pandas requests ftfy openpyxl

import os
import json
import time
import random
import requests
import pandas as pd
from datetime import datetime
from typing import Any, Dict, List, Optional

import ftfy

## 1) Configuración (token seguro vía config.py)

In [ ]:
# El token y los parámetros viven en config.py (IGNORADO por git).
# Primera vez:  cp config.example.py config.py  y coloca tu ELSEVIER_API_KEY.
import importlib
import config
importlib.reload(config)   # recarga si editaste config.py sin reiniciar el kernel

API_KEY    = config.ELSEVIER_API_KEY
INST_TOKEN = getattr(config, "ELSEVIER_INSTTOKEN", None)
assert API_KEY and API_KEY != "PUT-YOUR-KEY-HERE", \
    "Falta tu key: copia config.example.py a config.py y coloca ELSEVIER_API_KEY."

START_YEAR = getattr(config, "START_YEAR", 2019)
END_YEAR   = getattr(config, "END_YEAR", 2025)
OUT_DIR    = getattr(config, "OUT_DIR", "scopus_out")
DATA_DIR   = getattr(config, "DATA_DIR", "data")

SEARCH_URL = "https://api.elsevier.com/content/search/scopus"

# Throttling y límites de seguridad
SLEEP_MIN, SLEEP_MAX = 0.2, 0.6
PER_PAGE = 25                    # máximo típico del Search API
MAX_RECORDS_PER_AUTHOR = 5000    # tope por autor

os.makedirs(OUT_DIR, exist_ok=True)
print("Config cargada:", datetime.now().isoformat())

## 2) Roster de profesores

In [ ]:
# 21 profesores rastreados (TEC-001..TEC-021) con su Scopus Author ID.
authors = pd.DataFrame([
    {"prof_id": "TEC-001", "prof_name": "Molina-Perez, Edmundo", "author_id": "56709531800"},
    {"prof_id": "TEC-002", "prof_name": "Stein, Ernesto H.", "author_id": "7202194915"},
    {"prof_id": "TEC-003", "prof_name": "Bernal-Serrano, Daniel", "author_id": "57215937382"},
    {"prof_id": "TEC-004", "prof_name": "Aguilar-Gomez, Sandra", "author_id": "57205638710"},
    {"prof_id": "TEC-005", "prof_name": "Campos-Rivera, Paola Abril", "author_id": "57190304109"},
    {"prof_id": "TEC-006", "prof_name": "Garcia, Magdalena", "author_id": "60054037400"},
    {"prof_id": "TEC-007", "prof_name": "Duran-Fernandez, Roberto", "author_id": "55349505800"},
    {"prof_id": "TEC-008", "prof_name": "Santos, Miguel Angel", "author_id": "57015987800"},
    {"prof_id": "TEC-009", "prof_name": "Popper, Steven W.", "author_id": "7003635400"},
    {"prof_id": "TEC-010", "prof_name": "Peraza-Mues, Gonzalo G.", "author_id": "54997963100"},
    {"prof_id": "TEC-011", "prof_name": "Sobrino, Fernanda", "author_id": "59573511100"},
    {"prof_id": "TEC-012", "prof_name": "Benavides-Rincon, Guillermina", "author_id": "57216153915"},
    {"prof_id": "TEC-013", "prof_name": "Morales-Arilla, Jose", "author_id": "57282638100"},
    {"prof_id": "TEC-014", "prof_name": "Contreras-Loya, David", "author_id": "55971186800"},
    {"prof_id": "TEC-015", "prof_name": "Ponce-Lopez, Roberto", "author_id": "57205704459"},
    {"prof_id": "TEC-016", "prof_name": "Gomez-Zaldivar, Fernando", "author_id": "57219596780"},
    {"prof_id": "TEC-017", "prof_name": "Syme, James", "author_id": "57216353171"},
    {"prof_id": "TEC-018", "prof_name": "Diaz-Dominguez, Alejandro", "author_id": "55581628300"},
    {"prof_id": "TEC-019", "prof_name": "Silverio-Murillo, Adan", "author_id": "57212534001"},
    {"prof_id": "TEC-020", "prof_name": "Villarreal, Héctor J.", "author_id": "36855894400"},
    {"prof_id": "TEC-021", "prof_name": "De Unánue, Adolfo", "author_id": "56013629800"},
])
authors

## 3) Helpers: limpieza de texto, Scopus Search y OpenAlex

In [ ]:
# ── Limpieza de texto: arregla mojibake (p.ej. "MÃ©xico" -> "México") ─────────
def clean_text(x: Any) -> Optional[str]:
    if x is None:
        return None
    s = ftfy.fix_text(str(x)).strip()
    return s or None

# ── HTTP con decodificación UTF-8 forzada (causa raíz del mojibake) ──────────
def _get_json(url: str, headers: Dict[str, str], params: Optional[Dict[str, Any]] = None,
              timeout: int = 60):
    r = requests.get(url, headers=headers, params=params or {}, timeout=timeout)
    r.raise_for_status()
    # Scopus y OpenAlex sirven UTF-8; decodificamos los bytes crudos como UTF-8
    # en vez de dejar que requests adivine el charset (origen del mojibake).
    return json.loads(r.content.decode("utf-8"))

def _api_headers() -> Dict[str, str]:
    headers = {"X-ELS-APIKey": API_KEY, "Accept": "application/json"}
    if INST_TOKEN:
        headers["X-ELS-Insttoken"] = INST_TOKEN
    return headers

def sleep_a_bit():
    time.sleep(SLEEP_MIN + random.random() * (SLEEP_MAX - SLEEP_MIN))

# ── Scopus Search API ────────────────────────────────────────────────────────
def scopus_search(query: str, start: int = 0, count: int = 25, *, timeout: int = 60):
    params = {"query": query, "start": start, "count": count}
    return _get_json(SEARCH_URL, _api_headers(), params, timeout)

def build_author_query(author_id: str, start_year: int, end_year: int) -> str:
    # Años inclusivos
    return f"AU-ID({author_id}) AND PUBYEAR > {start_year-1} AND PUBYEAR < {end_year+1}"

def fetch_author_publications(author_id: str, start_year: int, end_year: int,
                              per_page: int = PER_PAGE,
                              max_records: int = MAX_RECORDS_PER_AUTHOR):
    query = build_author_query(author_id, start_year, end_year)
    js = scopus_search(query, start=0, count=1)
    total = int(js["search-results"]["opensearch:totalResults"])
    capped = min(total, max_records)
    entries: List[Dict[str, Any]] = []
    for start in range(0, capped, per_page):
        js = scopus_search(query, start=start, count=per_page)
        entries.extend(js["search-results"].get("entry", []))
        sleep_a_bit()
    return entries, {"total": total, "capped_total": capped, "query": query}

# ── OpenAlex API (gratis, sin credenciales) ──────────────────────────────────
OPENALEX_BASE = "https://api.openalex.org"
OPENALEX_HEADERS = {"User-Agent": "bibliometrics-research/1.0 (mailto:fuentesrivasfabian@gmail.com)"}

def fetch_openalex_work(doi: Optional[str], timeout: int = 30) -> Optional[Dict[str, Any]]:
    """Consulta OpenAlex por DOI; retorna el 'work' o None."""
    if not doi:
        return None
    doi_clean = str(doi).strip()
    for p in ("https://doi.org/", "http://dx.doi.org/", "https://dx.doi.org/", "http://doi.org/"):
        if doi_clean.lower().startswith(p):
            doi_clean = doi_clean[len(p):]
            break
    if not doi_clean:
        return None
    try:
        return _get_json(f"{OPENALEX_BASE}/works/doi:{doi_clean}", OPENALEX_HEADERS, timeout=timeout)
    except Exception:
        return None

def openalex_authors_affiliations(work: Optional[Dict[str, Any]]):
    """De un 'work' de OpenAlex devuelve (autores_ordenados, afiliaciones_únicas)."""
    if not work:
        return None, None
    authorships = work.get("authorships") or []
    names: List[str] = []
    inst_names: List[str] = []
    seen = set()
    for auth in authorships:
        nm = clean_text((auth.get("author") or {}).get("display_name"))
        if nm:
            names.append(nm)
        for inst in (auth.get("institutions") or []):
            iname = clean_text(inst.get("display_name"))
            if iname and iname not in seen:
                seen.add(iname)
                inst_names.append(iname)
    return ("; ".join(names) or None), ("; ".join(inst_names) or None)

## 4) Construir la base slim (una fila por publicación)

In [ ]:
def _scopus_fields(entry: Dict[str, Any]) -> Dict[str, Any]:
    """Campos a nivel paper desde una entrada del Search API (con fallback de afiliación)."""
    ident = str(entry.get("dc:identifier", "")).replace("SCOPUS_ID:", "")
    cover = clean_text(entry.get("prism:coverDate"))
    year = None
    if cover:
        try:
            year = int(str(cover)[:4])
        except Exception:
            year = None
    aff = entry.get("affiliation")
    aff_names: List[str] = []
    if isinstance(aff, list):
        for a in aff:
            if isinstance(a, dict):
                nm = clean_text(a.get("affilname"))
                if nm and nm not in aff_names:
                    aff_names.append(nm)
    return {
        "scopus_id": ident or None,
        "doi": clean_text(entry.get("prism:doi")),
        "title": clean_text(entry.get("dc:title")),
        "cover_date": cover,
        "year": year,
        "scopus_creator": clean_text(entry.get("dc:creator")),       # solo 1er autor (fallback)
        "scopus_affiliations": "; ".join(aff_names) or None,         # afiliaciones paper (fallback)
    }

def build_publication_dataset(authors_df: pd.DataFrame, start_year: int, end_year: int) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []   # uno por (paper, profesor rastreado)
    logs: List[Dict[str, Any]] = []
    for _, row in authors_df.iterrows():
        author_id = str(row["author_id"]); prof_id = row["prof_id"]; prof_name = row["prof_name"]
        print(f"Fetching AU-ID({author_id}) {prof_id} ...")
        entries, meta = fetch_author_publications(author_id, start_year, end_year)
        logs.append({"author_id": author_id, "prof_id": prof_id, "prof_name": prof_name, **meta})
        for e in entries:
            rec = _scopus_fields(e)
            if not rec["scopus_id"]:
                continue
            rec["prof_id"] = prof_id
            rec["prof_name"] = clean_text(prof_name)
            rows.append(rec)

    pd.DataFrame(logs).to_csv(os.path.join(OUT_DIR, "fetch_log.csv"),
                              index=False, encoding="utf-8-sig")

    raw = pd.DataFrame(rows)
    if raw.empty:
        return raw

    # ── Enriquecer cada paper ÚNICO con OpenAlex (cache: 1 llamada por paper) ──
    enrich: Dict[str, Any] = {}
    for _, r in raw.drop_duplicates("scopus_id").iterrows():
        work = fetch_openalex_work(r["doi"])
        enrich[r["scopus_id"]] = openalex_authors_affiliations(work)
        time.sleep(0.15)   # rate-limit cortés con OpenAlex

    # ── Deduplicar por publicación; profes rastreados como lista ─────────────
    slim: List[Dict[str, Any]] = []
    for scopus_id, grp in raw.groupby("scopus_id"):
        first = grp.iloc[0]
        a_str, aff_str = enrich.get(scopus_id, (None, None))
        slim.append({
            "tracked_prof_ids":   "; ".join(sorted(grp["prof_id"].dropna().unique())),
            "tracked_prof_names": "; ".join(sorted(grp["prof_name"].dropna().unique())),
            "scopus_id":    scopus_id,
            "doi":          first["doi"],
            "title":        first["title"],
            "cover_date":   first["cover_date"],
            "year":         first["year"],
            "authors":      a_str or first["scopus_creator"],          # OpenAlex, o 1er autor
            "affiliations": aff_str or first["scopus_affiliations"],   # OpenAlex, o paper-level
        })
    out = (pd.DataFrame(slim)
             .sort_values(["year", "tracked_prof_ids"], na_position="last")
             .reset_index(drop=True))
    return out

In [ ]:
pubs = build_publication_dataset(authors, START_YEAR, END_YEAR)
print("Shape:", pubs.shape)
pubs.head(5)

## 5) Guardar la base limpia (UTF-8)

In [ ]:
out_path = os.path.join(OUT_DIR, "scopus_publications.csv")
# utf-8-sig: abre correctamente en Excel y es compatible con pandas/Tableau
pubs.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Guardado:", out_path, "—", len(pubs), "publicaciones")

## 6) Validaciones

In [ ]:
# 1) Sin mojibake en las columnas de texto
text_cols = ["title", "authors", "affiliations", "tracked_prof_names"]
moji_sigs = ["Ã©", "Ã³", "Ã¡", "Ãº", "Ã­", "Ã±", "Ã ", "Ã\x81", "Â", "â€", "ï¿½", "\ufffd"]
blob = pubs[text_cols].fillna("").astype(str).agg(" ".join, axis=1).str.cat(sep=" ")
found = [m for m in moji_sigs if m in blob]
assert not found, f"Mojibake detectado: {found}"
print("✓ Sin mojibake en", text_cols)

# 2) Cobertura
print("Publicaciones únicas:", len(pubs))
print("Con lista de autores:", int(pubs["authors"].notna().sum()))
print("Con afiliaciones:    ", int(pubs["affiliations"].notna().sum()))
print("Sin DOI (no enriquecibles):", int(pubs["doi"].isna().sum()))
if len(pubs):
    print("Años cubiertos:", pubs["year"].min(), "-", pubs["year"].max())